# RAG Tools — Unit Tests
Step-by-step testing of chunk_document(), index_openapi_reference(), and retrieve_chunks().
Run each cell independently to inspect intermediate outputs and logs.

**Pre-requisites before running:**
1. Qdrant container must be running (`docker run -p 6333:6333 qdrant/qdrant`)
2. `data/references/openapi_reference/openapi_reference.md` must exist (run `fetch_openapi_reference.py`)

In [ ]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from config import get_logger
from config.settings import (
    OPENAPI_COLLECTION_NAME,
    CHUNK_SIZE, CHUNK_OVERLAP,
    EMBEDDING_MODEL, EMBEDDING_DIM,
    OPENAPI_REFERENCE_RETRIEVE_CHUNKS,
)
from tools.document_tools import load_openapi_reference
from tools.rag_tools import chunk_document, index_openapi_reference, search_openapi_reference, _get_client

logger = get_logger(__name__)
logger.info("Imports OK.")
logger.info(f"Collection : {OPENAPI_COLLECTION_NAME}")
logger.info(f"Chunk size : {CHUNK_SIZE} chars  |  overlap: {CHUNK_OVERLAP} chars")
logger.info(f"Embedding  : {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")
logger.info(f"Retrieve k : {OPENAPI_REFERENCE_RETRIEVE_CHUNKS} chunk(s)")

In [2]:
# Step 2 — Guard: check Qdrant is reachable
# Connects to the Qdrant server and lists existing collections.
# If this fails, start the Qdrant container before continuing.

client = _get_client()
collections = [c.name for c in client.get_collections().collections]

logger.info(f"Qdrant reachable. Existing collections: {collections}")
logger.info(
    f"OpenAPI reference collection ('{OPENAPI_COLLECTION_NAME}') "
    + ("already exists." if OPENAPI_COLLECTION_NAME in collections else "does NOT exist yet.")
)

2026-03-30 00:05:15 [INFO] __main__: Qdrant reachable. Existing collections: ['openapi_reference', '3gpp_rel18_28', '3gpp_rel18_28_28532']
2026-03-30 00:05:15 [INFO] __main__: OpenAPI reference collection ('openapi_reference') already exists.


In [3]:
# Step 3 — Test chunk_document()
# Loads the OpenAPI reference and splits it into overlapping chunks.
# Verifies that chunks are non-empty strings within the expected size range.

text = load_openapi_reference()
assert isinstance(text, str) and len(text) > 0, "OpenAPI reference is empty — run fetch_openapi_reference.py first."

chunks = chunk_document(text)

assert isinstance(chunks, list) and len(chunks) > 0
assert all(isinstance(c, str) and len(c) > 0 for c in chunks)

sizes = [len(c) for c in chunks]
logger.info(f"chunk_document OK — {len(chunks)} chunk(s) from {len(text)} chars.")
logger.info(f"  Min size : {min(sizes)} chars")
logger.info(f"  Max size : {max(sizes)} chars")
logger.info(f"  Avg size : {sum(sizes) // len(sizes)} chars")
logger.info(f"  Preview  : {chunks[0][:200]!r}")

2026-03-30 00:05:21 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-30 00:05:21 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-30 00:05:21 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-03-30 00:05:21 [INFO] __main__: chunk_document OK — 420 chunk(s) from 296246 chars.
2026-03-30 00:05:21 [INFO] __main__:   Min size : 34 chars
2026-03-30 00:05:21 [INFO] __main__:   Max size : 999 chars
2026-03-30 00:05:21 [INFO] __main__:   Avg size : 755 chars
2026-03-30 00:05:21 [INFO] __main__:   Preview  : '# OpenAPI Specification\n\n## Version 3.2.0\n\nThe key words "MUST", "MUST NOT", "REQUIRED", "SHALL", "SHALL NOT", "SHOULD", "SHOULD NOT", "RECOMMENDED", "NOT RECOMMENDED", "MAY", and "OPTIONAL" in this d'


In [6]:
# Step 4 — Test index_openapi_reference()
# Indexes the OpenAPI reference into Qdrant.
# Skips if the collection already has vectors (guard inside the function).
# Re-run with force=True to drop and reindex from scratch.

index_openapi_reference(force=False)

# Verify collection exists and has vectors
assert client.collection_exists(OPENAPI_COLLECTION_NAME), "Collection not found after indexing."
count = client.count(OPENAPI_COLLECTION_NAME).count
assert count > 0, "Collection is empty after indexing."

logger.info(f"index_openapi_reference OK — {count} vector(s) in '{OPENAPI_COLLECTION_NAME}'.")

2026-03-29 23:54:44 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-29 23:54:44 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-29 23:54:44 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-03-29 23:54:44 [INFO] tools.rag_tools: Chunked OpenAPI reference into 420 chunk(s).


2026-03-29 23:54:44 [DEBUG] tools.rag_tools: Dropped existing collection 'openapi_reference'.
2026-03-29 23:54:44 [INFO] tools.rag_tools: Collection 'openapi_reference' created (model=sentence-transformers/all-MiniLM-L6-v2, dim=384).
/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-29 23:54:51 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-03-29 23:54:51 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|██████████| 103/103

In [4]:
# Step 5 — Inspect collection info
# Shows collection status, vector count, and configuration.

info = client.get_collection(OPENAPI_COLLECTION_NAME)
logger.info(f"Collection   : {OPENAPI_COLLECTION_NAME}")
logger.info(f"Status       : {info.status}")
logger.info(f"Points count : {info.points_count:,}")
logger.info(f"Vectors config: {info.config.params.vectors}")

2026-03-30 00:05:34 [INFO] __main__: Collection   : openapi_reference
2026-03-30 00:05:34 [INFO] __main__: Status       : green
2026-03-30 00:05:34 [INFO] __main__: Points count : 420
2026-03-30 00:05:34 [INFO] __main__: Vectors config: {'text-dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}


In [5]:
# Step 6 — Test search_openapi_reference()
# Runs a semantic search query against the indexed collection.
# Verifies that the result is a non-empty string with the expected separator.

query = "HTTP methods and resource paths for NRM attributes"
result = search_openapi_reference(query, k=5)

assert isinstance(result, str) and len(result) > 0, "search_openapi_reference returned empty result."

chunks_retrieved = result.split("\n\n---\n\n")
logger.info(f"search_openapi_reference OK — {len(chunks_retrieved)} chunk(s) retrieved.")
logger.info(f"Query  : {query}")
for i, chunk in enumerate(chunks_retrieved):
    logger.info(f"  Chunk {i + 1} ({len(chunk)} chars): {chunk[:150]!r}")

2026-03-30 00:05:34 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-03-30 00:05:34 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9409.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-30 00:05:37 [DEBUG] tools.rag_tools: Retrieved 5 chunk(s) for: 'HTTP methods and resource paths for NRM attributes'
2026-03-30 00:05:37 [INFO] __main__: search_openapi_reference OK — 5 chunk(s) retrieved.
2026-03-30 00:05:37 [INFO] __main__: Query  : HTTP methods

In [6]:
chunks_retrieved

['##### Example: Ordered Multipart With Required Header\n\nAs described in [[?RFC2557]], a set of resources making up a web page can be sent in a `multipart/related` payload, preserving links from the `text/html` document to subsidiary resources such as scripts, style sheets, and images by defining a `Content-Location` header for each page.\nThe first part is used as the root resource (unless using `Content-ID`, which RFC2557 advises against and is forbidden in this example), so we use `prefixItems` and `prefixEncoding` to define that it must be an HTML resource, and then allow any of several different types of resources in any order to follow.\n\nThe `Content-Location` header is defined using `content: {text/plain: {...}}` to avoid percent-encoding its URI value; see [Appendix D](#appendix-d-serializing-headers-and-cookies) for further details.',
 '| <a name="oas-self"></a>$self | `string` | This string MUST be in the form of a URI reference as defined by [[RFC3986]] [Section 4.1](htt